# Module 1 — Brief 2 — Itération et amélioration du modèle

Ce notebook couvre les étapes du Brief 2 :

- Relecture du diagnostic du Brief 1
- Mise en oeuvre des actions correctives (dataset et/ou hyperparamètres)
- Réentraînement — Run 2
- Comparaison Run 1 vs Run 2 dans MLflow
- Choix et sauvegarde du meilleur modèle

Les cellules marquées `### A COMPLÉTER ###` contiennent du code partiel à terminer.
Les cellules sans marqueur sont fournies et peuvent être exécutées directement.

Prérequis : le notebook du Brief 1 a été exécuté, le Run 1 est disponible dans MLflow.

---
## 0. Imports et configuration

In [ ]:
import json
import torch
import mlflow
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

print("Imports OK")

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device utilisé : {device}")

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-1B"
MAX_LENGTH = 512

CATEGORIES_VALIDES = [
    "Support technique",
    "Demande commerciale",
    "Demande de transformation",
    "Réclamation",
    "Information générale"
]

---
## 1. Relecture du diagnostic du Brief 1

Avant toute action, relisez votre diagnostic du Brief 1.
Les actions de ce notebook doivent être directement motivées par ce diagnostic.
Une action corrective sans diagnostic est une intuition, pas une démarche.

In [ ]:
### A COMPLÉTER ###
# Objectif : résumer le diagnostic du Brief 1 avant d'agir
#
# Répondez aux questions suivantes dans des commentaires :
#
# 1. Quel était l'écart train_loss / eval_loss du Run 1 ?
# 2. Quelles catégories étaient mal classifiées ?
# 3. Le JSON généré était-il systématiquement valide ?
# 4. Quelle(s) action(s) corrective(s) avez-vous décidé d'appliquer ?
#    (Option A : dataset, Option B : hyperparamètres, Option C : les deux)

# 1. Ecart train_loss / eval_loss :
# ...

# 2. Catégories problématiques :
# ...

# 3. Qualité du JSON :
# ...

# 4. Action(s) corrective(s) choisie(s) et justification :
# ...

---
## 2. Action corrective — Option A : compléter le dataset

Si votre diagnostic a révélé que certaines catégories sont mal classifiées,
le dataset est probablement insuffisant sur ces cas.

Ajoutez des exemples ciblés sur les catégories ou types de demandes problématiques.

Si vous avez choisi l'Option B uniquement, passez directement à la section 3.

In [ ]:
# Chargement du dataset initial
with open("dataset_fastia_module1.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Dataset initial : {len(raw_data)} exemples")
print("\nDistribution initiale :")
for cat, count in Counter(e["output"]["categorie"] for e in raw_data).items():
    print(f"  {cat} : {count}")

In [ ]:
### A COMPLÉTER — Option A uniquement ###
# Objectif : ajouter des exemples ciblés sur les catégories problématiques
#
# 1. Identifier les catégories à renforcer d'après votre diagnostic
# 2. Rédiger entre 20 et 50 nouveaux exemples en respectant le format JSON du dataset
# 3. Chaque exemple doit avoir la structure :
#    { "input": "...", "output": { "categorie": "...", "priorite": "...", "reponse_suggeree": "..." } }
#
# Consignes de rédaction :
# - Ne pas copier des exemples existants
# - Varier le style : formel, informel, court, long
# - Couvrir des cas limites ou ambigus si vous en avez observés

nouveaux_exemples = [
    # A COMPLÉTER
]

print(f"Nouveaux exemples rédigés : {len(nouveaux_exemples)}")

In [ ]:
### A COMPLÉTER — Option A uniquement ###
# Objectif : fusionner le dataset initial avec les nouveaux exemples
#
# 1. Fusionner raw_data et nouveaux_exemples
# 2. Afficher la nouvelle taille du dataset
# 3. Afficher la nouvelle distribution des catégories
# 4. Vérifier que la distribution est plus équilibrée qu'avant
#
# 5. Sauvegarder le dataset enrichi dans dataset_fastia_v2.json
#
# Question : la distribution est-elle suffisamment équilibrée ?
# Un déséquilibre résiduel peut-il encore poser problème ?
# Répondez dans un commentaire.

raw_data_v2 = # A COMPLÉTER

print(f"Dataset enrichi : {len(raw_data_v2)} exemples")
print("\nNouvelle distribution :")
# A COMPLÉTER

with open("dataset_fastia_v2.json", "w", encoding="utf-8") as f:
    json.dump(raw_data_v2, f, ensure_ascii=False, indent=2)

print("\nDataset sauvegardé dans dataset_fastia_v2.json")

# Votre réponse :
# ...

---
## 3. Action corrective — Option B : ajuster les hyperparamètres

Si votre diagnostic a révélé un sous-apprentissage ou un sur-apprentissage,
ajustez les hyperparamètres en conséquence.

Rappel des signaux :
- Sous-apprentissage : loss élevée, prédictions incohérentes
  -> augmenter le learning rate ou le nombre d'époques
- Sur-apprentissage : train loss basse, eval loss haute
  -> réduire le learning rate, réduire les époques, augmenter lora_dropout

Si vous avez choisi l'Option A uniquement, conservez les hyperparamètres du Run 1.

In [ ]:
### A COMPLÉTER ###
# Objectif : définir les hyperparamètres du Run 2
#
# 1. Définir RUN2_CONFIG avec les valeurs ajustées
# 2. Pour chaque valeur modifiée, expliquer pourquoi dans un commentaire
#
# Si Option A uniquement : recopier les valeurs du Run 1 sans les modifier

RUN1_CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05
}

RUN2_CONFIG = {
    "learning_rate": # A COMPLÉTER,
    "num_train_epochs": # A COMPLÉTER,
    "per_device_train_batch_size": # A COMPLÉTER,
    "lora_r": # A COMPLÉTER,
    "lora_alpha": # A COMPLÉTER,
    "lora_dropout": # A COMPLÉTER,
    "run_name": "run2"
}

# Justification des changements :
# ...

---
## 4. Préparation du dataset pour le Run 2

On reconstruit le pipeline de préparation des données avec le dataset
approprié selon l'action corrective choisie.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer chargé")

In [ ]:
def build_prompt(exemple):
    instruction = (
        "Tu es un assistant FastIA. "
        "Analyse la demande suivante et réponds uniquement en JSON valide "
        "avec les champs : categorie, priorite, reponse_suggeree.\n\n"
        f"Demande : {exemple['input']}"
    )
    reponse = json.dumps(exemple["output"], ensure_ascii=False)
    return f"<s>[INST] {instruction} [/INST] {reponse} </s>"


def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
### A COMPLÉTER ###
# Objectif : préparer le dataset pour le Run 2
#
# 1. Charger le bon dataset selon votre action corrective :
#    - Option A ou C : charger dataset_fastia_v2.json
#    - Option B uniquement : charger dataset_fastia_module1.json
#
# 2. Appliquer build_prompt à tous les exemples
# 3. Tokeniser avec .map()
# 4. Diviser en train (80%) et test (20%)
# 5. Afficher la taille des deux jeux

dataset_path = # A COMPLÉTER : choisir le bon fichier

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

prompts = # A COMPLÉTER
dataset = Dataset.from_dict({"text": prompts})
dataset_tokenise = # A COMPLÉTER
split = # A COMPLÉTER

train_dataset = split["train"]
test_dataset = split["test"]

print(f"Entraînement : {len(train_dataset)} exemples")
print(f"Test         : {len(test_dataset)} exemples")

---
## 5. Chargement du modèle et configuration LoRA — Run 2

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Modèle de base chargé")

In [ ]:
### A COMPLÉTER ###
# Objectif : configurer LoRA avec les paramètres du Run 2
#
# Utiliser les valeurs de RUN2_CONFIG pour lora_r, lora_alpha, lora_dropout

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=# A COMPLÉTER,
    lora_alpha=# A COMPLÉTER,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=# A COMPLÉTER,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

---
## 6. Entraînement — Run 2

In [ ]:
### A COMPLÉTER ###
# Objectif : lancer le Run 2 dans la même expérience MLflow que le Run 1
#
# 1. Utiliser mlflow.set_experiment("fastia-llama-finetuning") — même expérience
# 2. Nommer le run avec RUN2_CONFIG["run_name"]
# 3. Loguer tous les paramètres de RUN2_CONFIG
#    Ajouter également : dataset_path, dataset_size, max_length
# 4. Lancer l'entraînement
# 5. Loguer train_loss et eval_loss
#
# Le output_dir doit être different du Run 1 : ./model_run2

mlflow.set_experiment("fastia-llama-finetuning")

with mlflow.start_run(run_name=RUN2_CONFIG["run_name"]):
    # A COMPLÉTER
    pass

---
## 7. Comparaison Run 1 vs Run 2

Les deux runs sont dans la même expérience MLflow.
On peut maintenant les comparer directement.

In [ ]:
# Récupération des deux runs depuis MLflow
runs = mlflow.search_runs(
    experiment_names=["fastia-llama-finetuning"],
    order_by=["start_time ASC"]
)

colonnes = [
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.num_train_epochs",
    "params.dataset_size",
    "params.lora_r",
    "params.lora_dropout",
    "metrics.train_loss",
    "metrics.eval_loss"
]

print(runs[colonnes].to_string(index=False))

In [ ]:
### A COMPLÉTER ###
# Objectif : comparer les prédictions qualitatives des deux runs
#
# 1. Définir la fonction predict() — identique au Brief 1
#
# 2. Charger le modèle du Run 1 depuis ./model_run1
#    Indice : PeftModel.from_pretrained(base_model, "./model_run1")
#
# 3. Charger le modèle du Run 2 depuis ./model_run2
#
# 4. Sur les mêmes 5 demandes problématiques identifiées dans le Brief 1,
#    comparer les prédictions des deux runs côte à côte :
#    Demande | Prédiction Run 1 | Prédiction Run 2
#
# 5. Les erreurs observées en Brief 1 ont-elles été corrigées ?
#    Répondez dans un commentaire.

def predict(texte, model, tokenizer, max_new_tokens=150):
    # A COMPLÉTER : reprendre la fonction du Brief 1
    pass


# Chargement des deux modèles
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model_run1 = # A COMPLÉTER
model_run2 = # A COMPLÉTER

# Demandes problématiques du Brief 1
demandes_problematiques = [
    # A COMPLÉTER : reprendre les cas qui posaient problème en Brief 1
]

for demande in demandes_problematiques:
    pred_run1 = predict(demande, model_run1, tokenizer)
    pred_run2 = predict(demande, model_run2, tokenizer)
    print(f"Demande : {demande}")
    print(f"Run 1   : {pred_run1}")
    print(f"Run 2   : {pred_run2}")
    print("---")

# Votre analyse :
# ...

---
## 8. Choix et sauvegarde du meilleur modèle

Le choix du meilleur modèle ne se fait pas uniquement sur la loss.
Une eval_loss légèrement plus haute peut être acceptable si les prédictions
qualitatives sont meilleures sur les cas problématiques.

In [ ]:
### A COMPLÉTER ###
# Objectif : justifier le choix du meilleur run et sauvegarder le modèle
#
# 1. Indiquer quel run vous choisissez : Run 1 ou Run 2
#
# 2. Justifier ce choix en vous appuyant sur :
#    - Les métriques MLflow (train_loss, eval_loss)
#    - La comparaison qualitative des prédictions
#    - L'impact de votre action corrective
#
# 3. Sauvegarder le meilleur modèle dans ./model_final
#    Indice : model.save_pretrained("./model_final")
#             tokenizer.save_pretrained("./model_final")
#
# 4. Vérifier le contenu du dossier sauvegardé

MEILLEUR_RUN = # A COMPLÉTER : "run1" ou "run2"
SAVE_PATH = "./model_final"

# Justification :
# ...

# Sauvegarde
meilleur_model = # A COMPLÉTER : model_run1 ou model_run2
# A COMPLÉTER : save_pretrained
# A COMPLÉTER : tokenizer.save_pretrained

print(f"Modèle sauvegardé dans {SAVE_PATH}")

---
## 9. Bilan de l'itération

Ce bilan sera repris dans le README.md du Brief 2.

In [ ]:
### A COMPLÉTER ###
# Objectif : produire un bilan structuré de l'itération
#
# Répondez aux questions suivantes dans des commentaires :
#
# 1. L'action corrective a-t-elle eu l'effet attendu ?
#    Comparez les métriques et prédictions avant / après.
#
# 2. Y a-t-il des cas problématiques qui persistent malgré la correction ?
#    Si oui, quelle en est la cause probable ?
#
# 3. Si vous deviez faire un Run 3, quelle action corrective appliqueriez-vous ?
#    (Cette question est ouverte, il n'y a pas de Run 3 dans ce brief)
#
# 4. Le modèle final est-il prêt à être exposé via une API ?
#    Quels sont ses points forts et ses limites connues ?

# 1. Effet de l'action corrective :
# ...

# 2. Cas persistants :
# ...

# 3. Run 3 hypothétique :
# ...

# 4. Prêt pour la production :
# ...

---
## Récapitulatif

Etapes réalisées dans ce notebook :

1. Relecture et synthèse du diagnostic du Brief 1
2. Mise en oeuvre de l'action corrective (dataset et/ou hyperparamètres)
3. Préparation du dataset pour le Run 2
4. Configuration LoRA avec les paramètres ajustés
5. Entraînement Run 2 tracké dans MLflow
6. Comparaison Run 1 vs Run 2 : métriques et prédictions qualitatives
7. Choix et sauvegarde du meilleur modèle
8. Bilan de l'itération

Prochaine étape — Brief 3 : conteneurisation avec Docker, logging Loguru,
tests Pytest et documentation de déploiement.